In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from pathlib import Path

# Dynamically find repo root (works from any notebook location)
ROOT = Path.cwd()
while not (ROOT / "UTILS").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MERGED = ROOT / "data" / "merged_dataset"

# # Load dataset
df = pd.read_parquet(MERGED / "unified_dataset.parquet")
df.head(10)
df


# Check that timestamp_ns is monotonic within each dataset subject session
ts_sorted = df.sort_values(["dataset", "subject_id", "session_id", "timestamp_ns"])

monotonic_flags = (
    ts_sorted
    .groupby(["dataset", "subject_id", "session_id"])["timestamp_ns"]
    .apply(lambda s: s.is_monotonic_increasing)
)

non_monotonic_groups = monotonic_flags[~monotonic_flags].reset_index()
print("Groups with non monotonic timestamps")
print(non_monotonic_groups)

# Find where both global and dataset activity ids are 9000
mask_9000 = (df["global_activity_id"] == 9000) & (df["dataset_activity_id"] == 9000)
df_9000 = df[mask_9000]

# Summary of which datasets and activity labels hit 9000
activity_9000_summary = (
    df_9000
    .groupby(["dataset", "global_activity_label", "dataset_activity_label"])
    .size()
    .reset_index(name="count_rows")
)

print(activity_9000_summary)


dt = (
    ts_sorted
    .groupby(["dataset", "subject_id", "session_id"])["timestamp_ns"]
    .diff()
)

dt_no_na = dt.dropna()

print("Summary of time steps in nanoseconds")
print(dt_no_na.describe())

print("Most common step sizes")
print(dt_no_na.value_counts().head(10))

# For a fifty Hz stream expect about twenty million nanoseconds
expected_step = 20_000_000  # adjust if you truly want a different spacing

bad_steps = dt_no_na[dt_no_na != expected_step]
print("Number of steps that differ from expected")
print(len(bad_steps))

if len(bad_steps) > 0:
    bad_info = (
        ts_sorted.loc[bad_steps.index, ["dataset", "subject_id", "session_id", "timestamp_ns"]]
        .head(20)
    )
    print("Example rows with unexpected spacing")
    print(bad_info)

sensor_cols = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]

print(df[sensor_cols].describe(percentiles=[0.01, 0.5, 0.99]))

# flag very large magnitudes that might indicate unit issues or spikes
for col in sensor_cols:
    big_mask = df[col].abs() > 200
    print(f"{col} rows with abs value greater than 200", big_mask.sum())

In [ ]:
import pandas as pd

# Load unified dataset
df = pd.read_parquet(MERGED / "unified_dataset.parquet")

# Keep only pamap2 and sort
pamap = (
    df[df["dataset"] == "pamap2"]
    .sort_values(["subject_id", "session_id", "timestamp_ns"])
    .copy()
)

# Time difference inside each subject session
pamap["dt"] = (
    pamap
    .groupby(["subject_id", "session_id"])["timestamp_ns"]
    .diff()
)

# Where the activity label changes inside a subject session
activity_change = (
    pamap["global_activity_id"]
    != pamap.groupby(["subject_id", "session_id"])["global_activity_id"].shift()
)

# Rows where the activity changed but timestamp did not
bad_changes = pamap.loc[
    activity_change & pamap["dt"].notna() & (pamap["dt"] == 0),
    ["subject_id", "session_id", "timestamp_ns", "global_activity_id", "dt"]
]

print("Number of activity changes where timestamp did not change")
print(len(bad_changes))
print(bad_changes.head(20))

# Optional session level summary to see if any subject session is affected
session_violations = (
    bad_changes
    .groupby(["subject_id", "session_id"])
    .size()
    .reset_index(name="n_bad_changes")
)

print("Sessions with any bad activity timestamp changes")
print(session_violations)

In [ ]:
import pandas as pd
import numpy as np

path = MERGED / "unified_dataset.parquet"
df = pd.read_parquet(path)

# identify the bad slice
mask_bad = (
    (df["dataset"] == "pamap2")
    & (df["subject_id"] == "S01")
    & (df["session_id"] == "Optional")
)

bad = df.loc[mask_bad].sort_values("timestamp_ns")

print("Rows in bad session:", len(bad))
print("Original ts head:")
print(bad["timestamp_ns"].head(10))

# build a synthetic 50 Hz clock for THIS session only
dt = 20_000_000  # 20 ms in ns for 50 Hz
n = len(bad)

# choice 1: make this session start at 0
new_ts = np.arange(n, dtype="int64") * dt

# choice 2: preserve original starting offset (if you care)
# start = bad["timestamp_ns"].iloc[0]
# new_ts = start + np.arange(n, dtype="int64") * dt

# write back
df.loc[bad.index, "timestamp_ns"] = new_ts

# re-check QA
ts_sorted = df.sort_values(["dataset", "subject_id", "session_id", "timestamp_ns"])
dt_all = (
    ts_sorted
    .groupby(["dataset", "subject_id", "session_id"])["timestamp_ns"]
    .diff()
    .dropna()
)

print("New global step stats:")
print(dt_all.describe())
print("Most common steps:")
print(dt_all.value_counts().head(10))

# sanity on that one session
dt_bad = (
    df.loc[mask_bad]
      .sort_values("timestamp_ns")["timestamp_ns"]
      .diff()
      .dropna()
)

print("Bad session step stats after fix:")
print(dt_bad.describe())
print(dt_bad.value_counts().head(5))

# if happy, overwrite parquet
df.to_parquet(path)
print("Patched unified_dataset.parquet with fixed PAMAP2 S01 Optional timestamps.")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet(MERGED / "unified_dataset.parquet")

expected_step = 20_000_000  # nanoseconds

pam = (
    df[df["dataset"] == "pamap2"]
    .sort_values(["subject_id", "session_id", "timestamp_ns"])
    .copy()
)

# one start time per session
starts = (
    pam
    .groupby(["subject_id", "session_id"])["timestamp_ns"]
    .agg(["min", "max", "count"])
    .reset_index()
)

print("PAMAP2 session start stats")
print(starts.head(20))

# per session time step checks
pam["dt"] = (
    pam
    .groupby(["subject_id", "session_id"])["timestamp_ns"]
    .diff()
)

# ignore first row in each session where dt is NaN
pam_dt = pam.dropna(subset=["dt"])

session_step_stats = (
    pam_dt
    .groupby(["subject_id", "session_id"])["dt"]
    .agg(["count", "min", "max", "mean"])
    .reset_index()
)

print("\nPer session step stats")
print(session_step_stats.head(20))

# any session that has a step different from the expected value
bad_sessions = (
    pam_dt[pam_dt["dt"] != expected_step]
    .groupby(["subject_id", "session_id"])
    .size()
    .reset_index(name="num_bad_steps")
)

print("\nSessions with any non standard step")
print(bad_sessions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#df = pd.read_parquet(MERGED / "unified_dataset.parquet")

target_dataset = "recofit"
fs = 50.0
window_seconds = 10.0
window_samples = int(window_seconds * fs)

sub = df[df["dataset"] == target_dataset].copy()
sub = sub.sort_values(["subject_id", "session_id", "timestamp_ns"])

# pick a session with enough samples
session_df = None
for (sid, sess), g in sub.groupby(["subject_id", "session_id"]):
    if len(g) >= window_samples:
        session_df = g
        subject_id = sid
        session_id = sess
        break

if session_df is None:
    raise ValueError("no session with enough samples")

segment = session_df.iloc[:window_samples].copy()

# time axis from sample index
n = np.arange(len(segment))
t = n / fs

ax = segment["acc_x"].to_numpy()
ay = segment["acc_y"].to_numpy()
az = segment["acc_z"].to_numpy()

gx = segment["gyro_x"].to_numpy()
gy = segment["gyro_y"].to_numpy()
gz = segment["gyro_z"].to_numpy()

acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)

# remove mean so gravity and constant bias do not dominate
acc_mag_zm = acc_mag - acc_mag.mean()
gyro_mag_zm = gyro_mag - gyro_mag.mean()

# raw time series
plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(t, ax, label="acc_x")
plt.plot(t, ay, label="acc_y")
plt.plot(t, az, label="acc_z")
plt.xlabel("time seconds")
plt.ylabel("acc m per s^2")
plt.title(f"{target_dataset} accel xyz ten second window")
plt.legend(loc="upper right")

plt.subplot(2, 1, 2)
plt.plot(t, gx, label="gyro_x")
plt.plot(t, gy, label="gyro_y")
plt.plot(t, gz, label="gyro_z")
plt.xlabel("time seconds")
plt.ylabel("gyro rad per s")
plt.title(f"{target_dataset} gyro xyz ten second window")
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

# spectrograms
plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.specgram(acc_mag_zm, NFFT=16, Fs=fs, noverlap=8)
plt.xlabel("time seconds")
plt.ylabel("frequency Hz")
plt.title(f"{target_dataset} accel magnitude spectrogram")
plt.colorbar(label="power")

plt.subplot(2, 1, 2)
plt.specgram(gyro_mag_zm, NFFT=32, Fs=fs, noverlap=16)
plt.xlabel("time seconds")
plt.ylabel("frequency Hz")
plt.title(f"{target_dataset} gyro magnitude spectrogram")
plt.colorbar(label="power")

plt.tight_layout()
plt.show()

In [ ]:

gyro_cols = ["gyro_x", "gyro_y", "gyro_z"]

# per dataset counts and completeness
g = df.groupby("dataset")

out = pd.DataFrame({
    "rows": g.size(),
    "gyro_x_nonnull": g["gyro_x"].count(),
    "gyro_y_nonnull": g["gyro_y"].count(),
    "gyro_z_nonnull": g["gyro_z"].count(),
})

out["gyro_x_pct"] = 100.0 * out["gyro_x_nonnull"] / out["rows"]
out["gyro_y_pct"] = 100.0 * out["gyro_y_nonnull"] / out["rows"]
out["gyro_z_pct"] = 100.0 * out["gyro_z_nonnull"] / out["rows"]

# how many rows have all three gyro axes present
out["gyro_all3_nonnull"] = g.apply(lambda x: x[gyro_cols].notna().all(axis=1).sum())
out["gyro_all3_pct"] = 100.0 * out["gyro_all3_nonnull"] / out["rows"]

print(out.sort_values("gyro_all3_pct", ascending=False))

In [ ]:
sub = sub.sort_values(["subject_id", "session_id", "timestamp_ns"])
counts = sub.groupby(["subject_id", "session_id"]).size().sort_values(ascending=False)

print("num sessions", len(counts))
print("top 10 session lengths")
print(counts.head(10))
print("max session length", counts.iloc[0] if len(counts) else None)


In [ ]:
import os
import numpy as np
import pandas as pd

# paths
UNIFIED_PARQUET = str(MERGED / "unified_dataset.parquet")
OUT_PARQUET = str(MERGED / "unified_dataset_small.parquet")

# sampling
ROWS_PER_DATASET = 5000
SEED = 42

# required columns in your contract
REQ_COLS = [
    "dataset","subject_id","session_id","timestamp_ns",
    "acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z",
    "global_activity_id","global_activity_label",
    "dataset_activity_id","dataset_activity_label",
]

# read only needed cols
df = pd.read_parquet(UNIFIED_PARQUET, columns=REQ_COLS)

# keep deterministic
rng = np.random.default_rng(SEED)

# per-dataset row sampling
parts = []
for ds, g in df.groupby("dataset", sort=False):
    n = len(g)
    k = min(int(ROWS_PER_DATASET), n)
    if k == 0:
        continue
    take = rng.choice(n, size=k, replace=False)
    parts.append(g.iloc[take])

small = pd.concat(parts, ignore_index=True)

# optional stable sort to preserve your primary index contract
small = small.sort_values(["dataset","subject_id","session_id","timestamp_ns"]).reset_index(drop=True)

# sanity prints
print("original rows", len(df))
print("small rows", len(small))
print("\nrows per dataset in small")
print(small["dataset"].value_counts())

# write
os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)
small.to_parquet(OUT_PARQUET, index=False)
print("\nwrote", OUT_PARQUET)

In [ ]:
import pandas as pd
from pathlib import Path

# ---- settings ----
parquet_path = MERGED / "unified_dataset.parquet"
expected_hz = 50  # matches config/default.yaml
probe_dataset = "samosa"  # lowercase, so we can drop it if needed
top_k = 10  # how many sessions with the largest gaps to preview
# ------------------

if not parquet_path.exists():
    raise FileNotFoundError(parquet_path)

expected_period_ns = 1e9 / expected_hz
gap_cutoff_ns = 1e9  # ≈1 second; adjust to your preferred threshold

cols = [
    "dataset",
    "subject_id",
    "session_id",
    "timestamp_ns",
]

df = pd.read_parquet(parquet_path, columns=cols).rename(
    columns=lambda c: c.strip().lower()
)
df = df[df["dataset"].str.strip().str.lower() != probe_dataset]

def session_gap_stats(g):
    g = g.sort_values("timestamp_ns")
    diffs = g["timestamp_ns"].diff().dropna()
    return pd.Series(
        {
            "samples": len(g),
            "max_gap_ns": diffs.max() if not diffs.empty else 0,
            "max_gap_seconds": (diffs.max() or 0) / 1e9,
            "num_large_gaps": (diffs > gap_cutoff_ns).sum(),
        }
    )

session_stats = (
    df.groupby(["dataset", "subject_id", "session_id"], sort=False)
      .apply(session_gap_stats)
      .reset_index()
      .sort_values("max_gap_ns", ascending=False)
)

print("Sessions with gaps above the cutoff:")
display(session_stats[session_stats["num_large_gaps"] > 0].head(top_k))

print("\nOverall gap distribution:")
display(
    session_stats["max_gap_seconds"]
    .describe(percentiles=[0.5, 0.9, 0.99])
    .to_frame(name="seconds")
)

# Optional: drill into one specific session
example = session_stats.iloc[0]
mask = (
    (df["dataset"] == example["dataset"])
    & (df["subject_id"] == example["subject_id"])
    & (df["session_id"] == example["session_id"])
)
example_series = df.loc[mask, "timestamp_ns"].sort_values().diff().dropna()
print(
    f"\nLargest gaps for session {tuple(example[['dataset','subject_id','session_id']])}:"
)
print(example_series.sort_values(ascending=False).head(10) / 1e9, "seconds")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

inPath = MERGED / "unified_dataset.parquet"
outPath = MERGED / "unified_dataset_clean.parquet"

expectedHz = 50
periodNs = int(round(1e9 / expectedHz))

largeGapCutoffSeconds = 1.0
largeGapCutoffNs = int(round(largeGapCutoffSeconds * 1e9))

smallInterpLimitSeconds = 0.2
smallInterpLimitSteps = int(round(smallInterpLimitSeconds * expectedHz))

toleranceNs = periodNs // 2

cols = [
    "dataset",
    "subject_id",
    "session_id",
    "timestamp_ns",
    "acc_x",
    "acc_y",
    "acc_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "global_activity_id",
    "global_activity_label",
    "dataset_activity_id",
    "dataset_activity_label",
]

df = pd.read_parquet(inPath, columns=cols)
df = df.rename(columns=lambda c: c.strip().lower())

df["dataset"] = df["dataset"].astype(str).str.strip().str.lower()

for c in ["subject_id", "session_id", "global_activity_label", "dataset_activity_label"]:
    df[c] = df[c].astype(str)

sensorCols = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]
labelCols = ["global_activity_id","global_activity_label","dataset_activity_id","dataset_activity_label"]

def cleanGroup(g):
    g = g.sort_values("timestamp_ns").reset_index(drop=True)

    t = g["timestamp_ns"].to_numpy(dtype=np.int64)
    d = np.diff(t)
    splitMask = np.r_[False, d > largeGapCutoffNs]
    segId = splitMask.cumsum()

    pieces = []

    for k in np.unique(segId):
        seg = g.loc[segId == k].copy()
        if len(seg) < 5:
            continue

        t0 = int(seg["timestamp_ns"].iloc[0])
        t1 = int(seg["timestamp_ns"].iloc[-1])

        grid = np.arange(t0, t1 + periodNs, periodNs, dtype=np.int64)
        gridDf = pd.DataFrame({"timestamp_ns": grid})

        merged = pd.merge_asof(
            gridDf,
            seg,
            on="timestamp_ns",
            direction="nearest",
            tolerance=toleranceNs,
        )

        merged[sensorCols] = merged[sensorCols].interpolate(
            method="linear",
            limit=smallInterpLimitSteps,
            limit_direction="both",
        )

        merged[labelCols] = merged[labelCols].ffill()

        merged = merged.dropna(subset=["acc_x","acc_y","acc_z","global_activity_id","dataset_activity_id"])

        if k > 0:
            merged["session_id"] = merged["session_id"].astype(str) + f"_seg{int(k):03d}"

        pieces.append(merged)

    if not pieces:
        return pd.DataFrame(columns=g.columns)

    return pd.concat(pieces, ignore_index=True)

cleaned = (
    df.groupby(["dataset","subject_id","session_id"], sort=False, group_keys=False)
      .apply(cleanGroup)
      .reset_index(drop=True)
)

cleaned["timestamp_ns"] = cleaned["timestamp_ns"].astype("int64")
for c in sensorCols:
    cleaned[c] = cleaned[c].astype("float32")
cleaned["global_activity_id"] = cleaned["global_activity_id"].astype("int16")
cleaned["dataset_activity_id"] = cleaned["dataset_activity_id"].astype("int16")

cleaned = cleaned.sort_values(["dataset","subject_id","session_id","timestamp_ns"]).reset_index(drop=True)

cleaned.to_parquet(outPath, index=False)

print("Wrote", outPath)
print("Rows before", len(df), "rows after", len(cleaned))

In [ ]:
import pandas as pd
from pathlib import Path
from collections import Counter

def validate_sessions(
    parquet_path: str,
    sample_rate_hz: float = 50.0,
    window_seconds: float = 2.56,
    gap_multiplier: float = 1.5,
    dataset_col: str = "dataset",
    subject_col: str = "subject_id",
    session_col: str = "session_id",
    time_col: str = "timestamp_ns",
    required_acc: tuple = ("acc_x", "acc_y", "acc_z"),
    required_gyro: tuple = ("gyro_x", "gyro_y", "gyro_z"),
    label_col: str = "global_activity_id",
):
    path = Path(parquet_path)
    if not path.exists():
        raise FileNotFoundError(path)

    cols_needed = (
        [dataset_col, subject_col, session_col, time_col, label_col]
        + list(required_acc) + list(required_gyro)
    )
    df = pd.read_parquet(path, columns=cols_needed)
    missing = [c for c in cols_needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df[dataset_col] = df[dataset_col].astype(str).str.strip().str.lower()
    df[subject_col] = df[subject_col].astype(str)
    df[session_col] = df[session_col].astype(str)

    expected_ns = 1e9 / sample_rate_hz
    cutoff_ns = gap_multiplier * expected_ns
    window_size = int(round(sample_rate_hz * window_seconds))

    issues = []
    gap_hist = Counter()
    total_sessions = 0
    viable_sessions = 0

    for (ds, subj, sess), g in df.groupby([dataset_col, subject_col, session_col], sort=False):
        total_sessions += 1
        g = g.sort_values(time_col)
        ts = g[time_col].to_numpy()

        diffs = pd.Series(ts).diff().dropna()
        big_gaps = diffs[diffs > cutoff_ns]

        segments = []
        start = 0
        for idx in big_gaps.index:
            segments.append(g.iloc[start:idx])
            start = idx
        segments.append(g.iloc[start:])

        large_gap_seconds = big_gaps.max() / 1e9 if not big_gaps.empty else 0.0
        windows_in_session = sum(len(seg) >= window_size for seg in segments)

        if windows_in_session == 0:
            issues.append(
                {
                    "dataset": ds,
                    "subject": subj,
                    "session": sess,
                    "total_rows": len(g),
                    "max_gap_s": round(large_gap_seconds, 2),
                    "segments": len(segments),
                    "segments_ge_window": windows_in_session,
                }
            )
        else:
            viable_sessions += 1

        if not big_gaps.empty:
            gap_hist.update({ds: 1})

    summary = {
        "total_sessions": total_sessions,
        "sessions_with_windows": viable_sessions,
        "sessions_without_windows": total_sessions - viable_sessions,
        "percent_viable": (viable_sessions / total_sessions * 100) if total_sessions else 0.0,
        "gap_histogram": dict(gap_hist),
    }

    return summary, pd.DataFrame(issues)

# Example usage
summary, failures = validate_sessions("data/merged_dataset/unified_dataset_clean_v2.parquet", gap_multiplier=10000.0)
print(summary)
display(failures.head())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

p = Path("data/merged_dataset/unified_dataset_clean_v2.parquet")

sampleRateHz = 50.0
expectedNs = 1e9 / sampleRateHz
gapMultiplier = 1.5
cutoffNs = gapMultiplier * expectedNs
windowSeconds = 2.56
windowSize = int(round(sampleRateHz * windowSeconds))

cols = [
    "dataset","subject_id","session_id","timestamp_ns",
    "acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z",
    "global_activity_id","dataset_activity_id",
    "global_activity_label","dataset_activity_label",
]

df = pd.read_parquet(p, columns=cols)
df = df.rename(columns=lambda c: c.strip().lower())
df["dataset"] = df["dataset"].astype(str).str.strip().str.lower()
df["subject_id"] = df["subject_id"].astype(str)
df["session_id"] = df["session_id"].astype(str)

print("rows", len(df))
print("sessions", df.groupby(["dataset","subject_id","session_id"], sort=False).ngroups)
print(df["dataset"].value_counts().head(20))

sizes = df.groupby(["dataset","subject_id","session_id"], sort=False).size().rename("n").reset_index()
print("min session n", int(sizes["n"].min()))
print("sessions under window", int((sizes["n"] < windowSize).sum()))
print("n quantiles")
print(sizes["n"].quantile([0.01,0.05,0.5,0.95,0.99]).astype(int))

sensorCols = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]
naRows = df[sensorCols].isna().all(axis=1).mean() * 100
print("rows all sensors NaN percent", round(naRows, 4))

dupKeys = df.duplicated(["dataset","subject_id","session_id","timestamp_ns"]).mean() * 100
print("duplicate primary key percent", round(dupKeys, 6))

def session_dt_stats(g):
    t = g["timestamp_ns"].to_numpy(dtype=np.int64)
    if len(t) < 2:
        return pd.Series({"n": len(t), "median_dt": np.nan, "p99_dt": np.nan, "max_dt": np.nan, "big_gaps": 0})
    d = np.diff(t)
    d = d[d > 0]
    if len(d) == 0:
        return pd.Series({"n": len(t), "median_dt": np.nan, "p99_dt": np.nan, "max_dt": np.nan, "big_gaps": 0})
    return pd.Series({
        "n": len(t),
        "median_dt": float(np.median(d)),
        "p99_dt": float(np.quantile(d, 0.99)),
        "max_dt": float(d.max()),
        "big_gaps": int((d > cutoffNs).sum()),
    })

dt = (
    df.groupby(["dataset","subject_id","session_id"], sort=False)
      .apply(session_dt_stats)
      .reset_index()
)

print("sessions with any big gaps", int((dt["big_gaps"] > 0).sum()))
print("max gap seconds overall", float(dt["max_dt"].max() / 1e9))

rate = 1e9 / dt["median_dt"]
print("median rate hz quantiles")
print(rate.quantile([0.01,0.05,0.5,0.95,0.99]))

print("sessions outside 50 Hz tolerance 0.5 Hz",
      int(((rate < (50 - 0.5)) | (rate > (50 + 0.5))).sum()))

def label_purity_stats(g):
    ids = g["global_activity_id"].to_numpy()
    return pd.Series({
        "n": len(ids),
        "unique_labels": int(pd.Series(ids).nunique()),
        "dominant_frac": float(pd.Series(ids).value_counts(normalize=True).iloc[0]) if len(ids) else np.nan,
    })

pur = (
    df.groupby(["dataset","subject_id","session_id"], sort=False)
      .apply(label_purity_stats)
      .reset_index()
)

print("sessions with more than one global label", int((pur["unique_labels"] > 1).sum()))
print("dominant label frac quantiles")
print(pur["dominant_frac"].quantile([0.01,0.05,0.5,0.95,0.99]))


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

inPath = Path("data/merged_dataset/unified_dataset_clean.parquet")
outPath = Path("data/merged_dataset/unified_dataset_clean_v2.parquet")

sampleRateHz = 50
windowSeconds = 2.56
windowSize = int(round(sampleRateHz * windowSeconds))

cols = [
    "dataset",
    "subject_id",
    "session_id",
    "timestamp_ns",
    "acc_x",
    "acc_y",
    "acc_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "global_activity_id",
    "global_activity_label",
    "dataset_activity_id",
    "dataset_activity_label",
]

df = pd.read_parquet(inPath, columns=cols)
df = df.rename(columns=lambda c: c.strip().lower())

df["dataset"] = df["dataset"].astype(str).str.strip().str.lower()
df["subject_id"] = df["subject_id"].astype(str)
df["session_id"] = df["session_id"].astype(str)

print("rows in", len(df))
print("datasets top")
print(df["dataset"].value_counts().head(20))

badDatasetMask = df["dataset"].isin(["none", "nan", "null", ""])
print("rows with bad dataset value", int(badDatasetMask.sum()))

df = df.loc[~badDatasetMask].copy()

sizes = (
    df.groupby(["dataset", "subject_id", "session_id"], sort=False)
      .size()
      .rename("n")
      .reset_index()
)

short = sizes["n"] < windowSize
print("sessions total", len(sizes))
print("sessions under window", int(short.sum()))

if short.any():
    keepSessions = sizes.loc[~short, ["dataset", "subject_id", "session_id"]]
    keepKey = pd.MultiIndex.from_frame(keepSessions)
    dfKey = pd.MultiIndex.from_frame(df[["dataset", "subject_id", "session_id"]])
    df = df.loc[dfKey.isin(keepKey)].copy()

df = df.sort_values(["dataset", "subject_id", "session_id", "timestamp_ns"]).reset_index(drop=True)

sensorCols = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]
for c in sensorCols:
    df[c] = df[c].astype("float32")

df["timestamp_ns"] = df["timestamp_ns"].astype("int64")
df["global_activity_id"] = df["global_activity_id"].astype("int16")
df["dataset_activity_id"] = df["dataset_activity_id"].astype("int16")
df["global_activity_label"] = df["global_activity_label"].astype(str)
df["dataset_activity_label"] = df["dataset_activity_label"].astype(str)

print("rows out", len(df))
print("datasets top after")
print(df["dataset"].value_counts().head(20))

sizes2 = (
    df.groupby(["dataset", "subject_id", "session_id"], sort=False)
      .size()
      .rename("n")
      .reset_index()
)
print("sessions out", len(sizes2))
print("min session length", int(sizes2["n"].min()))
print("p01 p05 p50 p95 p99")
print(sizes2["n"].quantile([0.01, 0.05, 0.50, 0.95, 0.99]).astype(int))

df.to_parquet(outPath, index=False)
print("wrote", outPath)


In [ ]:
#ell: dataset validation suite (schema + gaps + window counts + ontology checks)
# Requirements: numpy, pandas, pyarrow/fastparquet (for read_parquet)

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# -------------------------
# USER SETTINGS (edit these)
# -------------------------
DATA_GLOB = str(MERGED / "unified_dataset_clean_v2.parquet")  # can be a glob like "datadrive/*.parquet"
SCHEMA_JSON_PATH = None  # e.g. "ontology_schema.json" (optional)
OUT_JSON_PATH = "dataset_checks.json"  # where to write the report
MAX_ROWS = None          # e.g. 2_000_000 (optional)
MAX_SESSIONS = None      # e.g. 500 (optional, speeds up session/window scan)
GAP_MULTIPLIER = None    # override (optional). If None, uses config-like default below.
MIN_WINDOWS_PER_CLASS = 50

# Repo-like config (edit if your column names differ)
CFG = {
    "paths": {"data_glob": DATA_GLOB, "out_dir": "."},
    "data": {
        "dataset_column": "dataset",
        "subject_column": "subject_id",
        "session_column": "session_id",
        "time_column": "timestamp_ns",
        "label_id_column": "global_activity_id",
        "label_name_column": "global_activity_label",
        "acc_columns": ["acc_x", "acc_y", "acc_z"],
        "gyro_columns": ["gyro_x", "gyro_y", "gyro_z"],
        "unknown_activity_id": 9000,
        "probe_dataset": "SAMoSA",
        "sample_rate_hz": 50,
        "window_seconds": 2.56,
        "window_hop_ratio": 0.5,
        "gap_cutoff_multiplier": 1.5,  # used if GAP_MULTIPLIER is None
    },
}


# -------------------------
# Helpers (same logic as script)
# -------------------------

def _load_data(cfg: dict, max_rows: Optional[int] = None) -> pd.DataFrame:
    glob_pat = cfg["paths"]["data_glob"]
    files = sorted(Path(".").glob(glob_pat)) if "*" in glob_pat else [Path(glob_pat)]
    if not files:
        raise FileNotFoundError(f"No files matched: {glob_pat}")
    dfs = [pd.read_parquet(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)
    if max_rows:
        df = df.head(max_rows)
    return df


def check_schema(df: pd.DataFrame, cfg: dict, schema_json: Optional[str] = None) -> dict:
    d = cfg["data"]
    required = (
        [d["dataset_column"], d["subject_column"], d["session_column"], d["time_column"]]
        + d["acc_columns"] + d["gyro_columns"]
        + [d["label_id_column"], d["label_name_column"]]
    )
    present = [c for c in required if c in df.columns]
    missing = [c for c in required if c not in df.columns]
    extra = [c for c in df.columns if c not in required]

    schema_contract = None
    if schema_json and Path(schema_json).exists():
        schema_contract = json.loads(Path(schema_json).read_text())

    return {
        "required_columns": required,
        "present": present,
        "missing": missing,
        "extra": extra,
        "schema_json_loaded": schema_contract is not None,
        "dtypes": {c: str(df[c].dtype) for c in present},
    }


def check_gaps_and_windows(
    df: pd.DataFrame,
    cfg: dict,
    max_sessions: Optional[int] = None,
    gap_multiplier: Optional[float] = None,
) -> dict:
    d = cfg["data"]
    ds_col = d["dataset_column"]
    subj_col = d["subject_column"]
    sess_col = d["session_column"]
    t_col = d["time_column"]
    hz = d["sample_rate_hz"]
    win_sec = d["window_seconds"]

    expected_ns = 1e9 / hz
    gm = gap_multiplier if gap_multiplier else d.get("gap_cutoff_multiplier", 1.5)
    cutoff_ns = gm * expected_ns
    win_size = int(round(hz * win_sec))

    groups = df.groupby([ds_col, subj_col, sess_col], sort=False)
    keys = list(groups.groups.keys())
    if max_sessions:
        keys = keys[:max_sessions]

    total = len(keys)
    viable = 0
    gap_sessions = 0
    details: List[dict] = []

    for key in keys:
        g = groups.get_group(key).sort_values(t_col)
        ts = g[t_col].to_numpy()
        diffs = np.diff(ts)
        big = diffs[diffs > cutoff_ns]

        segs = np.split(np.arange(len(g)), np.where(diffs > cutoff_ns)[0] + 1)
        wins = sum(len(s) >= win_size for s in segs)

        if wins > 0:
            viable += 1
        if len(big):
            gap_sessions += 1

        details.append({
            "key": str(key),
            "rows": len(g),
            "big_gaps": int(len(big)),
            "max_gap_s": round(float(big.max()) / 1e9, 3) if len(big) else 0.0,
            "segments": len(segs),
            "windows_ge_size": wins,
        })

    return {
        "total_sessions_scanned": total,
        "viable_sessions": viable,
        "sessions_with_big_gaps": gap_sessions,
        "window_size_samples": win_size,
        "gap_cutoff_ns": cutoff_ns,
        "details": details,
    }


def check_ontology(df: pd.DataFrame, cfg: dict) -> dict:
    d = cfg["data"]
    lid = d["label_id_column"]
    lnm = d["label_name_column"]
    unk = d["unknown_activity_id"]

    vc = df[lnm].value_counts(dropna=False).to_dict()
    mapped = int((df[lid] != unk).sum())
    total = len(df)

    pairs = df[[lid, lnm]].drop_duplicates()
    one2one = bool(pairs.groupby(lid)[lnm].nunique().le(1).all())

    windows_per_class = {}
    hz = d["sample_rate_hz"]
    win_sec = d["window_seconds"]
    win_size = int(round(hz * win_sec))

    for label, count in vc.items():
        windows_per_class[str(label)] = count // win_size

    return {
        "total_rows": total,
        "mapped_rows": mapped,
        "coverage_pct": round(100.0 * mapped / max(total, 1), 2),
        "unique_labels": len(vc),
        "label_distribution": {str(k): int(v) for k, v in vc.items()},
        "id_to_label_one_to_one": one2one,
        "windows_per_class": windows_per_class,
    }


# -------------------------
# Run all checks
# -------------------------
print("Loading data...")
df = _load_data(CFG, max_rows=MAX_ROWS)
print(f"Loaded {len(df):,} rows, {df.columns.tolist()}")

report = {}
report["schema"] = check_schema(df, CFG, schema_json=SCHEMA_JSON_PATH)
report["gaps_windows"] = check_gaps_and_windows(df, CFG, max_sessions=MAX_SESSIONS, gap_multiplier=GAP_MULTIPLIER)
report["ontology"] = check_ontology(df, CFG)

# ---- Summary ----
s = report["schema"]
g = report["gaps_windows"]
o = report["ontology"]

print(f"\n=== SCHEMA === missing={s['missing']}, extra={len(s['extra'])}")
print(f"=== GAPS   === scanned={g['total_sessions_scanned']}, viable={g['viable_sessions']}, with_gaps={g['sessions_with_big_gaps']}")
print(f"=== ONTO   === coverage={o['coverage_pct']}%, labels={o['unique_labels']}, 1:1={o['id_to_label_one_to_one']}")

low = {k: v for k, v in o["windows_per_class"].items() if v < MIN_WINDOWS_PER_CLASS}
if low:
    print(f"\n⚠ Classes with <{MIN_WINDOWS_PER_CLASS} windows: {low}")

Path(OUT_JSON_PATH).write_text(json.dumps(report, indent=2, default=str))
print(f"\nwrote {OUT_JSON_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

# Cameraman (512x512 uint8)
I = data.camera().astype(np.float64)

def T(I):
    # LI: 1D Laplacian along rows (axis 0), Dirichlet-style edges
    LI = -2 * I.copy()
    LI[1:, :] += I[:-1, :]
    LI[:-1, :] += I[1:, :]

    # IL: 1D Laplacian along cols (axis 1)
    IL = -2 * I.copy()
    IL[:, 1:] += I[:, :-1]
    IL[:, :-1] += I[:, 1:]

    return LI + IL

TI = T(I)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(I, cmap="gray")
ax[0].set_title("Original")
ax[0].axis("off")

# Use symmetric scaling so positive/negative edges show clearly
m = np.max(np.abs(TI))
ax[1].imshow(TI, cmap="gray", vmin=-m, vmax=m)
ax[1].set_title("T(I) = LI + IL")
ax[1].axis("off")

plt.tight_layout()
plt.savefig("cameraman_laplacian.png")
plt.show()

In [ ]:
import numpy as np

def Lmat(N):
    L = -2*np.eye(N)
    L += np.eye(N, k=1)
    L += np.eye(N, k=-1)
    return L

for N in [2,3,4,5,8,10]:
    L = Lmat(N)
    A = np.kron(np.eye(N), L) + np.kron(L, np.eye(N))
    s = np.linalg.svd(A, compute_uv=False)
    print(N, s.min())  # > 0 means invertible

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Domain
x = np.linspace(0, 1, 600)

# (1) straight line
y_line = -x

# (2) quadratic (hits endpoints exactly)
y_quad = -(x**2)

# (3) smooth spline through chosen nodes
try:
    from scipy.interpolate import CubicSpline
    xs = np.array([0.0, 0.25, 0.55, 0.85, 1.0])
    ys = np.array([0.0, -0.05, -0.55, -0.95, -1.0])
    spline = CubicSpline(xs, ys, bc_type="natural")
    y_spline = spline(x)
except Exception:
    # fallback (not as smooth), in case scipy isn't available
    xs = np.array([0.0, 0.25, 0.55, 0.85, 1.0])
    ys = np.array([0.0, -0.05, -0.55, -0.95, -1.0])
    y_spline = np.interp(x, xs, ys)

# (4) theoretical brachistochrone (cycloid) from (0,0) to (1,-1)
# cycloid: X = a (t - sin t), Y = -a (1 - cos t)
# Solve (t - sin t)/(1 - cos t) = 1 for t in (0, 2π)
def f(t):
    return (t - np.sin(t)) / (1 - np.cos(t)) - 1

lo, hi = 1e-6, 2*np.pi - 1e-6
for _ in range(80):  # bisection
    mid = 0.5*(lo + hi)
    if f(lo) * f(mid) <= 0:
        hi = mid
    else:
        lo = mid
t1 = 0.5*(lo + hi)
a = 1 / (1 - np.cos(t1))

t = np.linspace(0, t1, 600)
x_brach = a * (t - np.sin(t))
y_brach = -a * (1 - np.cos(t))

# (5) one additional admissible curve (sinusoidal perturbation of line; endpoints preserved)
y_extra = -x - 0.2*np.sin(np.pi*x)

# Plot
plt.figure(figsize=(7, 6))
plt.plot(x, y_line, label="straight line")
plt.plot(x, y_quad, label="quadratic")
plt.plot(x, y_spline, label="spline")
plt.plot(x_brach, y_brach, label="brachistochrone")
plt.plot(x, y_extra, label="extra")

plt.scatter([0, 1], [0, -1], s=30)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlim(0, 1)
plt.ylim(-1.05, 0.05)
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("problem5a_curves.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
eps = 1e-6

# Common x grid (avoid x=0)
x = np.linspace(eps, 1.0, 200000)

# Curves y(x) and y'(x)
y_line = -x
yp_line = -np.ones_like(x)

y_quad = -(x**2)
yp_quad = -2*x

# Cubic spline curve (choose nodes; keep y<0 for x>0)
from scipy.interpolate import CubicSpline
xs = np.array([0.0, 0.2, 0.45, 0.75, 1.0])
ys = np.array([0.0, -0.20, -0.55, -0.85, -1.0])
spline = CubicSpline(xs, ys, bc_type="natural")
y_spline = spline(x)
yp_spline = spline(x, 1)

# Extra curve
y_extra = -x - 0.2*np.sin(np.pi*x)
yp_extra = -1.0 - 0.2*np.pi*np.cos(np.pi*x)

def time_functional_from_xy(y, yp):
    # assumes y<0 on (0,1]
    integrand = np.sqrt((1.0 + yp**2) / (2.0*g*(-y)))
    return np.trapz(integrand, x)

T_line = time_functional_from_xy(y_line, yp_line)
T_quad = time_functional_from_xy(y_quad, yp_quad)
T_spline = time_functional_from_xy(y_spline, yp_spline)
T_extra = time_functional_from_xy(y_extra, yp_extra)

# Brachistochrone (cycloid) in param t: x(t)=a(t-sin t), y(t)=-a(1-cos t)
# Solve boundary (1,-1): (t1 - sin t1) = (1 - cos t1)
def f(t):
    return (t - np.sin(t)) - (1 - np.cos(t))

lo, hi = 1e-6, 2*np.pi - 1e-6
for _ in range(80):
    mid = 0.5*(lo + hi)
    if f(lo)*f(mid) <= 0:
        hi = mid
    else:
        lo = mid
t1 = 0.5*(lo + hi)
a = 1.0 / (1.0 - np.cos(t1))

t = np.linspace(eps, t1, 200000)
dxdt = a*(1 - np.cos(t))
dydt = -a*np.sin(t)
y_t = -a*(1 - np.cos(t))

integrand_t = np.sqrt((dxdt**2 + dydt**2) / (2.0*g*(-y_t)))
T_brach = np.trapz(integrand_t, t)

times = {
    "straight line": T_line,
    "quadratic": T_quad,
    "cubic spline": T_spline,
    "brachistochrone": T_brach,
    "extra": T_extra,
}

for k,v in times.items():
    print(f"{k:15s}  T ≈ {v:.6f} s")

best = min(times, key=times.get)
print("\nmin time:", best, "=", times[best])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed parameters from Question 1(d)
g = 9.81
ell = 1.0

# Time grid on [0, T] with 1200 points
T = 10.0
n_points = 1200
t = np.linspace(0, T, n_points)
dt = t[1] - t[0]

# Number of trajectories
n_traj = 100

# Sample initial conditions from the required ranges
np.random.seed(42)
theta0_deg = np.random.uniform(-80, 80, n_traj)
omega0_deg = np.random.uniform(-100, 100, n_traj)

# Convert to radians
theta0 = np.deg2rad(theta0_deg)
omega0 = np.deg2rad(omega0_deg)

# Equivalent first-order system:
# theta' = omega
# omega' = -(g/ell) sin(theta)
def f(z):
    theta, omega = z
    return np.array([omega, -(g / ell) * np.sin(theta)])

def rk4_step(z, dt):
    k1 = f(z)
    k2 = f(z + 0.5 * dt * k1)
    k3 = f(z + 0.5 * dt * k2)
    k4 = f(z + dt * k3)
    return z + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

# Solve 100 trajectories
Z = np.zeros((n_traj, n_points, 2))
for j in range(n_traj):
    Z[j, 0] = [theta0[j], omega0[j]]
    for i in range(n_points - 1):
        Z[j, i + 1] = rk4_step(Z[j, i], dt)

theta = Z[:, :, 0]
omega = Z[:, :, 1]

# Choose 5 representative trajectories
rep_idx = [0, 1, 2, 3, 4]

# Plot theta(t) and phase portrait
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for j in rep_idx:
    axes[0].plot(t, theta[j], lw=1.5, label=f"traj {j+1}")
axes[0].set_xlabel("t")
axes[0].set_ylabel(r"$\theta(t)$")
axes[0].set_title(r"Time evolution of $\theta(t)$")
axes[0].legend()

for j in rep_idx:
    axes[1].plot(theta[j], omega[j], lw=1.5, label=f"traj {j+1}")
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel(r"$\omega$")
axes[1].set_title(r"Phase portrait $(\theta,\omega)$")
axes[1].legend()

plt.tight_layout()
plt.savefig("pendulum_rk4_trajectories.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

# Assumes these already exist from 2(a):
# t            shape (n_points,)
# theta        shape (n_traj, n_points)
# omega        shape (n_traj, n_points)   from solving the first-order system

dt = t[1] - t[0]
n_traj, n_points = theta.shape

# Finite differences for acceleration alpha_FD = d omega / dt
alpha_fd = np.zeros_like(omega)

# Central differences for interior points
alpha_fd[:, 1:-1] = (omega[:, 2:] - omega[:, :-2]) / (2 * dt)

# One-sided differences at the boundaries
alpha_fd[:, 0] = (omega[:, 1] - omega[:, 0]) / dt
alpha_fd[:, -1] = (omega[:, -1] - omega[:, -2]) / dt

# Since we solved the first-order system, omega is already available.
# For consistency with the assignment notation:
omega_fd = omega.copy()

print("theta shape    :", theta.shape)
print("omega_fd shape :", omega_fd.shape)
print("alpha_fd shape :", alpha_fd.shape)

# Split trajectories into train / val / test = 70 / 15 / 15
rng = np.random.default_rng(42)
idx = rng.permutation(n_traj)

n_train = int(0.70 * n_traj)
n_val = int(0.15 * n_traj)
n_test = n_traj - n_train - n_val

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

data = {
    "train": {
        "theta": theta[train_idx],
        "omega": omega_fd[train_idx],
        "alpha": alpha_fd[train_idx],
        "t": t,
        "idx": train_idx,
    },
    "val": {
        "theta": theta[val_idx],
        "omega": omega_fd[val_idx],
        "alpha": alpha_fd[val_idx],
        "t": t,
        "idx": val_idx,
    },
    "test": {
        "theta": theta[test_idx],
        "omega": omega_fd[test_idx],
        "alpha": alpha_fd[test_idx],
        "t": t,
        "idx": test_idx,
    },
}

for split in ["train", "val", "test"]:
    print(
        split,
        data[split]["theta"].shape,
        data[split]["omega"].shape,
        data[split]["alpha"].shape,
    )

# Optional flattened version for later LNN training:
# each row is one time sample (theta_i, omega_i, alpha_i)
train_theta = data["train"]["theta"].reshape(-1, 1)
train_omega = data["train"]["omega"].reshape(-1, 1)
train_alpha = data["train"]["alpha"].reshape(-1, 1)

val_theta = data["val"]["theta"].reshape(-1, 1)
val_omega = data["val"]["omega"].reshape(-1, 1)
val_alpha = data["val"]["alpha"].reshape(-1, 1)

test_theta = data["test"]["theta"].reshape(-1, 1)
test_omega = data["test"]["omega"].reshape(-1, 1)
test_alpha = data["test"]["alpha"].reshape(-1, 1)

print("flattened train:", train_theta.shape, train_omega.shape, train_alpha.shape)
print("flattened val  :", val_theta.shape, val_omega.shape, val_alpha.shape)
print("flattened test :", test_theta.shape, test_omega.shape, test_alpha.shape)